<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


**<h1> 实验：使用 Faster R-CNN 进行目标检测 </h1>**


Faster R-CNN 是一种使用区域提议进行目标检测的方法。在本实验中，你将使用在 COCO 数据集上预训练的 Faster R-CNN。你将学习如何按名称检测多个对象，并使用目标预测正确的可能性。


预计所需时间：**30** 分钟


<h2>目标</h2>


使用 Faster R-CNN 进行目标检测，利用对象名称和/或对象的可能性对预定对象进行分类。


# 目录



本笔记本分为以下几个部分：

-   [导入库并定义辅助函数](#Import-Libraries-and-Define-Auxiliary-Functions)
-   [加载预训练 Faster R-CNN](#Load-Pre-trained-Faster-R-CNN)
-   [目标定位](#Object-Localization)
-   [目标检测](#Object-Detection)
-   [使用上传的图像测试模型](#Test-Model-With-An-Uploaded-Image)



----


 下载实验图像：**安装可能需要一些时间，请耐心等待。**


In [ ]:
%%time
%pip install numpy matplotlib opencv-python-headless
%pip install torch==2.8.0+cpu torchvision==0.23.0+cpu torchaudio==2.8.0+cpu \
--index-url https://download.pytorch.org/whl/cpu


In [ ]:
! wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-CV0101EN-Coursera/images%20/images_part_5/DLguys.jpeg
! wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-CV0101EN-Coursera/images%20/images_part_5/watts_photos2758112663727581126637_b5d4d192d4_b.jpeg
! wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-CV0101EN-Coursera/images%20/images_part_5/istockphoto-187786732-612x612.jpeg
! wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-CV0101EN-Coursera/images%20/images_part_5/jeff_hinton.png



## 导入库并定义辅助函数


深度学习库，可能需要更新：


In [ ]:
#! conda install pytorch=1.1.0 torchvision -c pytorch -y


In [ ]:
import torchvision
from torchvision import  transforms 
import torch
from torch import no_grad
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights


用于从网络获取数据的库  


In [ ]:
import requests


用于图像处理和可视化的库


In [ ]:
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt


## 🔍 技术原理：`get_predictions()` 的后处理逻辑

`get_predictions(pred, threshold, objects)` 在模型的原始输出之上做三道过滤：

1. **置信度阈值**：丢掉 `score < threshold` 的预测——阈值越高要求越严，误检越少但漏检可能增多（本实验后半部分会看到阈值设太低会「 hallucinate 」出不存在的对象）；
2. **类别筛选**：若指定 `objects`（如 `"person"`），只保留该类别的预测；
3. **名称映射**：把数字类别索引转换成可读字符串。

之后 `draw_box()` 把过滤后的框连同「类别 + 置信度」文本画到图像上。

该函数会为预测类别分配字符串名称，并删除置信度低于阈值的预测。


In [ ]:
def get_predictions(pred,threshold=0.8,objects=None ):
    """
    该函数会为预测类别分配字符串名称，并删除置信度低于阈值的预测 
    
    pred：每个元素包含对应不同对象信息的元组的列表；每个元素包括类别 yhat、属于该类别的概率以及对应对象边界框坐标的元组 
    image：冻结的表面
    predicted_classes：每个元素包含对应不同对象信息的元组的列表；每个元素包括类别名称、属于该类别的概率以及对应对象边界框坐标的元组 
    阈值
    """


    predicted_classes= [(COCO_INSTANCE_CATEGORY_NAMES[i],p,[(box[0], box[1]), (box[2], box[3])]) for i,p,box in zip(list(pred[0]['labels'].numpy()),pred[0]['scores'].detach().numpy(),list(pred[0]['boxes'].detach().numpy()))]
    predicted_classes=[  stuff  for stuff in predicted_classes  if stuff[1]>threshold ]
    
    if objects  and predicted_classes :
        predicted_classes=[ (name, p, box) for name, p, box in predicted_classes if name in  objects ]
    return predicted_classes


为每个对象绘制边界框


In [ ]:
def draw_box(predicted_classes, image, rect_th=10, text_size=3, text_th=3):
    """
    为每个对象绘制边界框。

    predicted_classes：每个元素包含以下内容的元组的列表：
        - 类别名称（字符串）
        - 概率（浮点数）
        - 边界框：((x1, y1), (x2, y2))
    image：表示图像的 torch 张量 (C, H, W)
    """

    # 将张量图像转换为 OpenCV 可用的 BGR uint8 格式
    img = (np.clip(cv2.cvtColor(
        np.clip(image.numpy().transpose((1, 2, 0)), 0, 1),
        cv2.COLOR_RGB2BGR), 0, 1) * 255).astype(np.uint8).copy()

    for predicted_class in predicted_classes:
        label = predicted_class[0]
        probability = predicted_class[1]
        box = predicted_class[2]

        pt1 = tuple(map(int, box[0]))  # 左上角
        pt2 = tuple(map(int, box[1]))  # 右下角

        cv2.rectangle(img, pt1, pt2, (0, 255, 0), rect_th)
        cv2.putText(img, f"{label}: {round(probability, 2)}", pt1,
                    cv2.FONT_HERSHEY_SIMPLEX, text_size, (0, 255, 0), thickness=text_th)

    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()


该函数通过释放内存来加速你的代码。


该函数会释放一些内存：


In [ ]:
def save_RAM(image_=False):
    global image, img, pred
    torch.cuda.empty_cache()
    del(img)
    del(pred)
    if image_:
        image.close()
        del(image)


## 🔍 技术原理：Faster R-CNN 是什么

Faster R-CNN（2015, [arXiv:1506.01497](https://arxiv.org/abs/1506.01497)）是经典的**两阶段（two-stage）深度目标检测器**，本实验使用 torchvision 在 COCO 数据集上预训练的 ResNet50-FPN 版本。与 Haar 级联手工设计特征不同，它的特征完全由**神经网络自动学习**。一次前向传播分两个阶段：

**阶段 1：RPN（Region Proposal Network，区域提议网络）**

- 主干网络（Backbone：ResNet50 + FPN 特征金字塔）提取多尺度特征图；
- RPN 在特征图的每个位置预设一组**锚框（anchor boxes）**——不同长宽比和尺度的参考框；
- 对每个锚框预测「是否包含物体」的分数和位置修正量，筛出几百个**可能包含物体的候选区域（RoI）**。

**阶段 2：RoI Head（分类与定位）**

- 每个候选区域通过 RoI Align 从特征图中裁剪出固定大小的特征；
- 两个并行分支分别输出：**类别分数**（91 类）和**边界框回归修正量**（对候选框位置的精细调整）；
- 同一物体可能产生多个重叠框，模型用**非极大值抑制NMS**保留最优框。

**与 Haar 级联对比**：Haar 靠人工特征 + 级联滑窗，速度快但对视角/光照敏感；Faster R-CNN 靠学习到的特征，鲁棒性好、能同时输出类别和边界框，但计算量大、推理慢（本实验特意安装 CPU 版 torch）。

## 加载预训练 Faster R-CNN


<a href='https://arxiv.org/abs/1506.01497'>Faster R-CNN</a> 是一种预测图像中潜在对象的边界框和类别分数的模型，已在 <a href="https://cocodataset.org/">COCO<a> 数据集上预训练。


In [ ]:
# 使用更新后的方式加载预训练权重
weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
model_ = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights=weights)
model_.eval()

# 冻结参数
for name, param in model_.named_parameters():
    param.requires_grad = False

print("完成")


该函数调用 Faster R-CNN <code>model_</code>，同时节省内存：


In [ ]:
def model(x):
    with torch.no_grad():
        yhat = model_(x)
    return yhat


## 🔍 技术原理：91 个类别与 COCO 数据集

- COCO 是微软发布的大规模目标检测数据集，含 80 个物体类别 + 1 个背景类 = **91 个索引**（索引 0 固定为 `__background__`，因此列表中间存在一些未使用的空位编号）。
- 模型输出的 `labels` 是**类别索引**，通过 `COCO_INSTANCE_CATEGORY_NAMES[index]` 映射为可读名称（例如索引 3 → `car`）。
- 因为是预训练权重，模型只能识别这 80 类物体；不在这份清单里的物体无法被正确分类，只能被错分为相近类别或背景。

以下是 91 个类别。


In [ ]:
COCO_INSTANCE_CATEGORY_NAMES = [
    '__background__', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
    'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'N/A', 'stop sign',
    'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow',
    'elephant', 'bear', 'zebra', 'giraffe', 'N/A', 'backpack', 'umbrella', 'N/A', 'N/A',
    'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball',
    'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket',
    'bottle', 'N/A', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl',
    'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza',
    'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'N/A', 'dining table',
    'N/A', 'N/A', 'toilet', 'N/A', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone',
    'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'N/A', 'book',
    'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush'
]
len(COCO_INSTANCE_CATEGORY_NAMES)


## 目标定位


在目标定位中，我们定位图像中对象的存在，并用边界框指示其位置。考虑 <a href="https://www.utoronto.ca/news/ai-fuels-boom-innovation-investment-and-jobs-canada-report-says">Geoffrey Hinton</a> 的图像


In [ ]:
img_path='jeff_hinton.png'
half = 0.5
image = Image.open(img_path)

image.resize( [int(half * s) for s in image.size] )

plt.imshow(image)
plt.show()


我们将创建一个变换对象以将图像转换为张量。


In [ ]:
transform = transforms.Compose([transforms.ToTensor()])


我们将图像转换为张量。


In [ ]:
img = transform(image)


## 🔍 技术原理：模型输出的是什么

`model([img])` 的输出是一个字典列表，每个字典包含三个张量：

- `boxes`：`(N, 4)` 边界框坐标，格式为 `(t, l, r, b)` = （上, 左, 下, 右），即矩形左上角和右下角的坐标；
- `labels`：`(N,)` 每个框的类别索引（通过 `COCO_INSTANCE_CATEGORY_NAMES` 映射为名称）；
- `scores`：`(N,)` 每个框的**置信度**，已按分数从高到低排序，且模型内部已完成 NMS 去重。

⚠️ 注意：`scores` 常被当作「概率 / likelihood」，但它实际是模型的置信度分数，并不严格等于概率。

我们可以进行预测，输出是一个字典，包含多个预测类别、属于该类别的概率以及对应类别边界框的坐标。


In [ ]:
pred = model([img])
print(f"pred: {pred}")

<b>注意</b>：如果你直接调用 <code>model_([img])</code>，它会使用更多内存


我们得到了 35 个不同的类别预测，按潜在对象的可能性分数排序。


In [ ]:
pred[0]['labels']


我们有每个类别的可能性：


In [ ]:
pred[0]['scores']


*注意* 这里我们将 likelihood 用作 probability 的同义词。许多神经网络输出的是属于某个特定类别的概率。这里输出的是预测的置信度，因此我们用 likelihood 来区分两者。


类别编号对应具有相应类别名称的列表索引


In [ ]:
index=pred[0]['labels'][0].item()
COCO_INSTANCE_CATEGORY_NAMES[index]


我们有边界框的坐标


In [ ]:
bounding_box=pred[0]['boxes'][0].tolist()
bounding_box


这些分量对应矩形的左上角和右下角，更精确地说：
<p>上（t）、左（l）、下（b）、右（r）</p>
我们需要对它们取整


In [ ]:
t,l,r,b=[round(x) for x in bounding_box]


我们将张量转换为 OpenCV 数组并绘制带有边界框的图像：


In [ ]:
img_plot=(np.clip(cv2.cvtColor(np.clip(img.numpy().transpose((1, 2, 0)),0,1), cv2.COLOR_RGB2BGR),0,1)*255).astype(np.uint8)
cv2.rectangle(img_plot,(t,l),(r,b),(0, 255, 0), 10) # 使用坐标绘制矩形
plt.imshow(cv2.cvtColor(img_plot, cv2.COLOR_BGR2RGB))
plt.show()
del img_plot, t, l, r, b


我们可以定位对象；我们使用 <code>get_predictions</code> 函数来实现。输入是预测结果 <code>pred</code> 和你想要定位的 <code>objects</code>。


In [ ]:
pred_class=get_predictions(pred,objects="person")
draw_box(pred_class, img)
del pred_class


我们可以设置阈值 <code>threshold</code>。这里我们设置阈值为 1，即 100% 的可能性。


In [ ]:
get_predictions(pred,threshold=1.0,objects="person")


由于可能性不是 100%，这里没有输出。让我们尝试阈值为 0.98，并使用 draw_box 函数绘制边界框以及类别及其四舍五入后的可能性。


In [ ]:
pred_thresh=get_predictions(pred,threshold=0.98,objects="person")
draw_box(pred_thresh,img)
del pred_thresh


删除对象以节省内存，我们将在每个单元格后运行此操作：


In [ ]:
save_RAM(image_=True)


我们可以定位多个对象，考虑以下 <a href='https://www.kdnuggets.com/2015/03/talking-machine-deep-learning-gurus-p1.html'>图像</a>，我们可以检测图像中的人。


In [ ]:
img_path='DLguys.jpeg'
image = Image.open(img_path)
image.resize([int(half * s) for s in image.size])
plt.imshow(np.array(image))
plt.show()


我们可以设置阈值来检测对象，0.9 似乎有效。


In [ ]:
img = transform(image)
pred = model([img])
pred_thresh=get_predictions(pred,threshold=0.8,)
draw_box(pred_thresh,img,rect_th= 1,text_size= 0.5,text_th=1)
del pred_thresh


或者我们可以使用 objects 参数：


In [ ]:

pred_obj=get_predictions(pred,objects="person")
draw_box(pred_obj,img,rect_th= 1,text_size= 0.5,text_th=1)
del pred_obj


如果阈值设置得太低，我们会检测到不存在的对象。


In [ ]:
pred_thresh=get_predictions(pred,threshold=0.01)
draw_box(pred_thresh,img,rect_th= 1,text_size= 0.5,text_th=1)
del pred_thresh


以下代码行将通过减少内存使用来加速你的代码。


In [ ]:
save_RAM(image_=True)


## 🔍 技术原理：目标定位 vs 目标检测

- **目标定位（Localization）**：图像中**主体基本只有一个对象**，任务只是确定它在哪、是什么——本节用 Hinton 的照片，只找 `person`；
- **目标检测（Detection）**：图像中**可能有多个不同类别的对象**，要把它们全部找出来并分别分类——本节会同时检测猫、狗、鸟等多个类别。

两者用的是同一个模型和同一次前向传播，区别只在于**后处理**时如何设置 `threshold` 和 `objects` 参数。

## 目标检测


在目标检测中，我们既要找出类别，也要检测图像中的对象。考虑以下 <a href="https://www.dreamstime.com/stock-image-golden-retriever-puppy-lying-parakeet-perched-its-head-weeks-old-next-to-british-shorthair-kitten-sitting-image30336051">图像</a>


In [ ]:
img_path='istockphoto-187786732-612x612.jpeg'
image = Image.open(img_path)
image.resize( [int(half * s) for s in image.size] )
plt.imshow(np.array(image))
plt.show()
del img_path


如果设置阈值，我们可以检测所有可能性高于该阈值的对象。


In [ ]:
img = transform(image)
pred = model([img])
pred_thresh=get_predictions(pred,threshold=0.97)
draw_box(pred_thresh,img,rect_th= 1,text_size= 1,text_th=1)
del pred_thresh


以下代码行将通过减少内存使用来加速你的代码。


In [ ]:
 save_RAM(image_=True)


我们可以指定想要分类的对象，例如猫和狗：


In [ ]:
img_path='istockphoto-187786732-612x612.jpeg'
image = Image.open(img_path)
img = transform(image)
pred = model([img])
pred_obj=get_predictions(pred,objects=["dog","cat"])
draw_box(pred_obj,img,rect_th= 1,text_size= 0.5,text_th=1)
del pred_obj



In [ ]:
#save_RAM()

如果阈值设置得太低，我们可能会检测到可能性较低的对象；这里我们将阈值设置为 0.7，结果错误地检测到了一只猫


In [ ]:
# img = transform(image)
# pred = model([img])
pred_thresh=get_predictions(pred,threshold=0.70,objects=["dog","cat"])
draw_box(pred_thresh,img,rect_th= 1,text_size= 1,text_th=1)
del pred_thresh


In [ ]:
save_RAM(image_=True)



我们还可以检测其他对象。考虑以下 <a href='https://www.flickr.com/photos/watts_photos/27581126637'>图像</a>；我们可以检测汽车和飞机


In [ ]:
img_path='watts_photos2758112663727581126637_b5d4d192d4_b.jpeg'
image = Image.open(img_path)
image.resize( [int(half * s) for s in image.size] )
plt.imshow(np.array(image))
plt.show()
del img_path


In [ ]:
img = transform(image)
pred = model([img])
pred_thresh=get_predictions(pred,threshold=0.997)
draw_box(pred_thresh,img)
del pred_thresh


In [ ]:
save_RAM(image_=True)


## 使用上传的图像测试模型


你可以输入图像的 URL，看看能否检测其中的对象。只需记住它必须有图像扩展名，例如 <code>jpg</code> 或 <code>png</code>。


In [ ]:
url='https://www.plastform.ca/wp-content/themes/plastform/images/slider-image-2.jpg'


我们将执行 GET 请求从网络下载图像，并将其转换为 RGB 图像。


In [ ]:
image = Image.open(requests.get(url, stream=True).raw).convert('RGB')
del url


In [ ]:
img = transform(image )
pred = model([img])
pred_thresh=get_predictions(pred,threshold=0.95)
draw_box(pred_thresh, img)
del pred_thresh


In [ ]:
save_RAM(image_=True)


上传你的图像，看看能否检测到对象
<p><b>Instructions on how to upload image:</b></p>
使用上传按钮从本地机器上传图像
<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-CV0101EN-SkillsNetwork/images/instruction.png" width="300">
</center>


替换为你目录中显示的图像名称


In [ ]:
img_path='images2.jpg'
image = Image.open(img_path) # 加载图像
plt.imshow(np.array(image ))
plt.show()


检测对象


In [ ]:
img = transform(image )
pred = model(img.unsqueeze(0))
pred_thresh=get_predictions(pred,threshold=0.95)
print(f"pred_thresh: {pred_thresh}")
draw_box(pred_thresh,img,rect_th= 1,text_size= 1,text_th=1)


<h2>作者</h2>


 [Joseph Santarcangelo]( https://www.linkedin.com/in/joseph-s-50398b136/) 拥有电气工程博士学位，他的研究专注于使用机器学习、信号处理和计算机视觉来确定视频如何影响人类认知。Joseph 自完成博士学位以来一直在 IBM 工作。


# 参考文献


[1]  图片来自：https://homepages.cae.wisc.edu/~ece533/images/
    
[2]  <a href='https://pillow.readthedocs.io/en/stable/index.html'>Pillow Docs</a>

[3]  <a href='https://opencv.org/'>Open CV</a>

[4] Gonzalez, Rafael C. 和 Richard E. Woods。"Digital image processing." (2017)。


<!-- # 变更日志
| 日期（YYYY-MM-DD） | 版本 | 修改人 | 变更描述      |
| ----------------- | ------- | ---------- | ----------------------- |
| 2025-07-14        | 1.0   | Sathya Priya| 将实验转换为 JupyterCurrent 笔记本 |-->


<!--<h2>变更日志</h2>-->


<!--<table>
    <tr>
        <th>日期（YYYY-MM-DD）</th>
        <th>版本</th>
        <th>修改人</th>
        <th>变更描述</th>
    </tr>
    <tr>
        <td>2020-07-20</td>
        <td>0.2</td>
        <td>Joseph Santarcangelo </td>
        <td>Modified Multiple Areas</td>
    </tr>
    <tr>
        <td>2020-07-17</td>
        <td>0.1</td>
        <td>Azim</td>
        <td>Created Lab Template</td>
    </tr>
</table>
-->



<h3 align="center"> &#169; IBM Corporation。保留所有权利。 <h3/>
